<a href="https://colab.research.google.com/github/abuwildanm/Information-Retrieval/blob/master/Projek.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projek Akhir
# Information Retrieval with Vector Space Model

---

## Nama Anggota Kelompok


1.   Abu Wildan Mucholladin
2.   Lukman Hakim W.
3.   Panji Gemilang






## Persiapan Data

In [ ]:
# Import Library Standard
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Import Library Sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB

# Import Library Sklearn (Evaluasi Tak Berperingkat)
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score


# Import Library Sklearn (Evaluasi Berperingkat)
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Import Library untuk Stemming
# !pip install Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

In [ ]:
# Read Dataset
data = pd.read_excel('Dataset\dataKumparan1.xlsx')
# data = pd.read_csv('dataKumparan2.csv')
data.head()

In [ ]:
# Label Mapping
# data['Topic'] = data['Topic'].map({'Politik':0, 'Teknologi':1, 'Travel':2})
# data.head()

In [ ]:
# Ukuran Dataset
print('Ukuran dataset:', data.shape)

In [ ]:
# Pembagian data training & testing
x_train, x_test, y_train, y_test = train_test_split(data['Content'], data['Topic'], train_size=0.5, test_size=0.16)

In [ ]:
train_data = pd.DataFrame({'teks':x_train, 'label':y_train})
test_data = pd.DataFrame({'teks':x_test, 'label':y_test})

In [ ]:
train_data.head()

In [ ]:
test_data.head()

In [ ]:
# Ukuran Data Training & Testing
print('Ukuran data train:', train_data.shape)
print('Ukuran data test:', test_data.shape)
n_train = train_data.shape[0]
n_test = test_data.shape[0]

In [ ]:
sparse_data = pd.concat([train_data, test_data], ignore_index=True)
sparse_data.head()

In [ ]:
# Ukuran Sparse Data
print('Ukuran sparse data:', sparse_data.shape)
n_document = sparse_data.shape[0]

In [ ]:
def normalisasi(kolom):
    normal = (kolom-kolom.min())/(kolom.max()-kolom.min())
    return normal

## Skenario 1 - Tanpa Stemming Dengan Stopword

### Preprocessing

### Perhitungan Bobot

In [ ]:
vectorizer = CountVectorizer()
tf = vectorizer.fit_transform(sparse_data['teks'])
print('Jumlah dokumen:', tf.shape[0])
print('Jumlah term:', tf.shape[1])

In [ ]:
print('Daftar Term:')
vectorizer.get_feature_names()

In [ ]:
print('Daftar Stopword:')
vectorizer.get_stop_words()

In [ ]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names())
tf_matrix

In [ ]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

In [ ]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

In [ ]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names())
weight_matrix

In [ ]:
# Pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

### Perhitungan Cosine Similarity

In [ ]:
# Perhitungan cosine similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity: ', cosim.shape)

In [ ]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [ ]:
# Baris = dokumen train, Kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

In [ ]:
# Cosim Matrix + Label
cosim_matrix['Label Train'] = train_data['label'].values
label_row = dict(zip(name, test_data['label'].values))
label_cosim = cosim_matrix.append(label_row, ignore_index=True)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)

label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_test

In [ ]:
label_cosim

### Evaluasi

In [ ]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = []
    peringkat = 0
    counter_relevant = 0
    
    for r in relevant:
      peringkat += 1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant / peringkat)
      
    # List of average precision
    average_precision.append(np.mean(precision))
  
  return average_precision
    
def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    precision_at_k.append(precision)
      
  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
  r_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    r_precision.append(precision)
      
  return r_precision

In [ ]:
print('Hasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@K: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))

## Skenario 2 - Tanpa Stemming Tanpa Stopword

In [ ]:
vectorizer = CountVectorizer()
tf = vectorizer.fit_transform(sparse_data['teks'])
print('Jumlah dokumen:', tf.shape[0])
print('Jumlah term:', tf.shape[1])

In [ ]:
print('Daftar Term:')
vectorizer.get_feature_names()

In [ ]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names())
tf_matrix

In [ ]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

In [ ]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

In [ ]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names())
weight_matrix

In [ ]:
# Pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

### Perhitungan Cosine Similarity

In [ ]:
# Perhitungan cosine similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity: ', cosim.shape)

In [ ]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [ ]:
# Baris = dokumen train, Kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

In [ ]:
# nanti disini coba normalisasi
# cosim_matrix = normalisasi(cosim_matrix.iloc[:, 0:10])

cosim_matrix

In [ ]:
# Cosim Matrix + Label
cosim_matrix['Label Train'] = train_data['label'].values
label_row = dict(zip(name, test_data['label'].values))
label_cosim = cosim_matrix.append(label_row, ignore_index=True)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)

label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_cosim

### Evaluasi

In [ ]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = []
    peringkat = 0
    counter_relevant = 0
    
    for r in relevant:
      peringkat += 1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant / peringkat)
      
    # List of average precision
    average_precision.append(np.mean(precision))
  
  return average_precision
    
def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    precision_at_k.append(precision)
      
  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
  r_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    r_precision.append(precision)
      
  return r_precision

In [ ]:
print('Hasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@K: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))

## Skenario 3 - Dengan Stemming Dengan Stopword

In [ ]:
# Create stemmer
stemmerFactory = StemmerFactory()
stemmer = stemmerFactory.create_stemmer()

# Stopword List
stopWordRemoverFactory = StopWordRemoverFactory()
stopwords = stopWordRemoverFactory.get_stop_words()

In [ ]:
# Stem Process
for row in range(n_document):
  sparse_data.loc[row, 'teks'] = stemmer.stem(sparse_data.loc[row, 'teks'])

In [ ]:
sparse_data.head()

### Perhitungan Bobot

In [ ]:
vectorizer = CountVectorizer(stop_words=stopwords)
tf = vectorizer.fit_transform(sparse_data['teks'])
print('Jumlah dokumen:', tf.shape[0])
print('Jumlah term:', tf.shape[1])

In [ ]:
print('Daftar Term:')
vectorizer.get_feature_names()

In [ ]:
print('Daftar Stopword:')
vectorizer.get_stop_words()

In [ ]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names())
tf_matrix

In [ ]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

In [ ]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

In [ ]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names())
weight_matrix

In [ ]:
# Pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

### Perhitungan Cosine Similarity

In [ ]:
# Perhitungan cosine similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity: ', cosim.shape)

In [ ]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [ ]:
# Baris = dokumen train, Kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

In [ ]:
# Cosim Matrix + Label
cosim_matrix['Label Train'] = train_data['label'].values
label_row = dict(zip(name, test_data['label'].values))
label_cosim = cosim_matrix.append(label_row, ignore_index=True)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)

label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_test

In [ ]:
label_cosim

### Evaluasi

In [ ]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = []
    peringkat = 0
    counter_relevant = 0
    
    for r in relevant:
      peringkat += 1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant / peringkat)
      
    # List of average precision
    average_precision.append(np.mean(precision))
  
  return average_precision
    
def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    precision_at_k.append(precision)
      
  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
  r_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    r_precision.append(precision)
      
  return r_precision

In [ ]:
print('Hasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@K: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))

## Skenario 4 - Dengan Stemming Tanpa Stopword

### Preprocessing

**Stemming**

In [ ]:
# Create stemmer
stemmerFactory = StemmerFactory()
stemmer = stemmerFactory.create_stemmer()

# Stopword List
stopWordRemoverFactory = StopWordRemoverFactory()
stopwords = stopWordRemoverFactory.get_stop_words()

In [ ]:
# Stem Process
for row in range(n_document):
  sparse_data.loc[row, 'teks'] = stemmer.stem(sparse_data.loc[row, 'teks'])

In [ ]:
sparse_data.head()

### Perhitungan Bobot

In [ ]:
vectorizer = CountVectorizer(stop_words=stopwords)
tf = vectorizer.fit_transform(sparse_data['teks'])
print('Jumlah dokumen:', tf.shape[0])
print('Jumlah term:', tf.shape[1])

In [ ]:
print('Daftar Term:')
vectorizer.get_feature_names()

In [ ]:
print('Matriks Tf:')
tf_matrix = pd.DataFrame(tf.toarray(), columns=vectorizer.get_feature_names())
tf_matrix

In [ ]:
print('Matriks Tf (khusus data train):')
tf_train = tf_matrix[:n_train]
tf_train.shape

In [ ]:
transformer = TfidfTransformer(sublinear_tf=True)

# Penyesuaian df agar query (data test) tidak dihitung pada perhitungan df
n = n_train
df = tf_train.astype(bool).sum(axis=0)
idf = np.log(n/df)
transformer.idf_ = idf

weight = transformer.fit_transform(tf)
print('Jumlah dokumen:', weight.shape[0])
print('Jumlah term:', weight.shape[1])

In [ ]:
weight_matrix = pd.DataFrame(weight.toarray(), columns=vectorizer.get_feature_names())
weight_matrix

In [ ]:
# Pembagian matriks bobot
weight_train = weight_matrix[:n_train]
weight_test = weight_matrix[n_train:]

### Perhitungan Cosine Similarity

In [ ]:
# Perhitungan cosine similarity
cosim = cosine_similarity(weight_train, weight_test)
print('Ukuran matriks cosine similarity: ', cosim.shape)

In [ ]:
name = []
for i in range(n_test):
  name.append('Dokumen ' + str(i))

In [ ]:
# Baris = dokumen train, Kolom = dokumen test
print('Matriks Cosine Similarity:')
cosim_matrix = pd.DataFrame(cosim, columns=name)
cosim_matrix.shape

In [ ]:
# Cosim Matrix + Label
cosim_matrix['Label Train'] = train_data['label'].values
label_row = dict(zip(name, test_data['label'].values))
label_cosim = cosim_matrix.append(label_row, ignore_index=True)
label_cosim.rename({label_cosim.index[-1]:'Label Test'}, inplace=True)

label_test = pd.DataFrame(label_cosim.iloc[-1])
label_test = label_test.T
label_test

In [ ]:
label_cosim

### Evaluasi

In [ ]:
def average_precision(cosim_matrix, n_retrieve):
  average_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = []
    peringkat = 0
    counter_relevant = 0
    
    for r in relevant:
      peringkat += 1
      if r == True:
        counter_relevant += 1
        precision.append(counter_relevant / peringkat)
      
    # List of average precision
    average_precision.append(np.mean(precision))
  
  return average_precision
    
def mean_average_precision(cosim_matrix, n_retrieve):
  ap = average_precision(cosim_matrix, n_retrieve)
  map = np.mean(ap)
  return map

def precision_at_k(cosim_matrix, k_retrieve):
  precision_at_k = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:k_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    precision_at_k.append(precision)
      
  return precision_at_k

def r_precision(cosim_matrix, n_retrieve):
  r_precision = []
  
  # Loop untuk mengambil setiap query di cosim matrix
  for column in cosim_matrix.iloc[:, :-1]:
    
    # Sort and get top n
    sorted_cosim = cosim_matrix.sort_values(column, ascending=False)
    top_n = sorted_cosim.iloc[:n_retrieve]
#     print(top_n)
    
    # List of relevance
    relevant = (np.array(top_n['Label Train']) == np.array(label_test[column]))
#     print(relevant)
    
    # List of precision
    precision = np.sum(relevant) / len(relevant)
    r_precision.append(precision)
      
  return r_precision

In [ ]:
print('Hasil Evaluasi MAP: ', mean_average_precision(cosim_matrix, 10))
print('Hasil Evaluasi Precision@K: ', precision_at_k(cosim_matrix, 10))
print('Hasil Evaluasi R-Precision: ', r_precision(cosim_matrix, 10))